In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.config('spark.driver.host','localhost').appName('05').getOrCreate()

In [3]:
yt = spark.read.csv('C:\\datas\\3rd_week_data\\youtubevideos.csv',
                   header = True,
                   inferSchema=True)

In [4]:
yt.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [5]:
yt.collect()[0]['publish_time']

'2017-11-13T17:13:01.000Z'

- 알파벳 `T`와 `Z`를 삭제한 후, `timestamp type`으로 변환

## 1. `publish_time` 열의 값을 `timestamp` type으로 변환하기 

- `T`는 공백으로 대체, `Z`는 삭제
- 특정 문자를 대체하는 함수: `regexp_replace(column, str1, str2)`
> - column의 문자열 중 str1을 str2로 대체
- `withColumn`과 함께 사용하여, 변환된 값을 새로운 이름의 열로 생성할 수 있음

In [6]:
# regexp_replace는 sql.functions 모듈에 있음
from pyspark.sql.functions import *

In [7]:
# yt의 publish_time의 T를 공백으로 대체한 후, p_time_tmp1 열을 생성함 ( yt_tmp1 )
yt_tmp1 = yt.withColumn('p_time_tmp1',regexp_replace(yt.publish_time,'T',' '))
# yt의 p_time_tmp1의 Z를 삭제한 후, p_time_tmp2 열을 생성함 ( yt_tmp2 )
yt_tmp2 = yt_tmp1.withColumn('p_time_tmp2',regexp_replace(yt_tmp1.p_time_tmp1,'Z',''))

In [8]:
print(yt_tmp2.select('publish_time','p_time_tmp1','p_time_tmp2').collect()[0][0])
print(yt_tmp2.select('publish_time','p_time_tmp1','p_time_tmp2').collect()[0][1])
print(yt_tmp2.select('publish_time','p_time_tmp1','p_time_tmp2').collect()[0][2])

2017-11-13T17:13:01.000Z
2017-11-13 17:13:01.000Z
2017-11-13 17:13:01.000


- `to_timestamp(column,시간이 저장된 형태)`: column값에 날짜/시간이 저장된 형태를 미리 알려주고, `timestamp`로 변환
> - `y`: 연도, `M`: 월, `d`: 일, `H`: 시간(0~23), `m`: 분, `s`: 초, `S`: 초(소수점)
> - ex> `MM`: `09` <-> `M`: `9`, `HH`: `23` <-> `hh`: `11`

In [9]:
yt_tmp3 = yt_tmp2.withColumn('real_p_time',to_timestamp(yt_tmp2.p_time_tmp2,'yyyy-MM-dd HH:mm:ss.SSS'))

In [10]:
yt_tmp3.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- p_time_tmp1: string (nullable = true)
 |-- p_time_tmp2: string (nullable = true)
 |-- real_p_time: timestamp (nullable = true)



### **sol2>** `translate(column, str1, str2)`를 사용
- column의 값에서 str1을 str2로 대체함
- `str1` 과 `str2`는 char(문자 하나)가 하나씩 매칭되어 대체됨
> - ex> `ABC`, `abc`의 경우, **A -> a, B -> b, C -> c**로 대체
> - ex> `ABC`, `ac`의 경우, **A -> a, B -> c, C -> 삭제**로 대체
> - `translate(column, 'TZ',' ')`를 실행할 경우, `T`는 공백으로 대체되고, `Z`는 삭제됨

In [11]:
yt_new = yt.withColumn('publish_time',to_timestamp(translate(yt.publish_time,'TZ',' '), 'yyyy-MM-dd HH:mm:ss.SSS'))
yt_new.select('publish_time').show(5)
yt_new.printSchema()

+-------------------+
|       publish_time|
+-------------------+
|2017-11-13 17:13:01|
|2017-11-13 07:30:00|
|2017-11-12 19:05:24|
|2017-11-13 11:00:04|
|2017-11-12 18:01:41|
+-------------------+
only showing top 5 rows

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [13]:
yt = yt.withColumn('publish_time',translate(yt.publish_time,'TZ',' '))
yt = yt.withColumn('publish_time',to_timestamp(yt.publish_time, 'yyyy-MM-dd HH:mm:ss.SSS'))
yt.select('publish_time').show(5)
yt.printSchema()

+-------------------+
|       publish_time|
+-------------------+
|2017-11-13 17:13:01|
|2017-11-13 07:30:00|
|2017-11-12 19:05:24|
|2017-11-13 11:00:04|
|2017-11-12 18:01:41|
+-------------------+
only showing top 5 rows

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



- `trending_date` 열의 값을 `date` type으로 변경
> - 'timestamp'는 **연월일시분초** `date`는 **연월일**
- `to_date(column,연월일이 표현된 방식)`: `yyyy-MM-dd` 형태로 반환

In [14]:
yt = yt.withColumn('trending_date',to_date(yt.trending_date,'yy.dd.MM'))
yt.select('trending_date').show(3)
yt.printSchema()

+-------------+
|trending_date|
+-------------+
|   2017-11-14|
|   2017-11-14|
|   2017-11-14|
+-------------+
only showing top 3 rows

root
 |-- video_id: string (nullable = true)
 |-- trending_date: date (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



## 2. 새로운 **Column 생성**

- `when(조건식, val1).otherwhise(val2)`: 조건식을 만족할 경우 val1, 만족하지 않는 경우 val2 값을 가지는 column 생성
- `like`가 `dislikes` 보다 많으면 **good**, 나머지 경우, **bad**를 갖는 `favor`열 생성

In [15]:
yt = yt.withColumn('favor', when(yt.likes > yt.dislikes,'good').otherwise('bad'))
yt.select('likes','dislikes','favor').show(5)
# likes와 dislikes가 str타입이므로 ASCII 값을 비교하여 조건식을 판단함

+------+--------+-----+
| likes|dislikes|favor|
+------+--------+-----+
| 57527|    2966| good|
| 97185|    6146| good|
|146033|    5339|  bad|
| 10172|     666|  bad|
|132235|    1989|  bad|
+------+--------+-----+
only showing top 5 rows



- `likes`와 `dislikes`가 str 타입이므로 `int` 타입으로 변환한 후, 위를 다시 수행
- `column.cast(type)`

In [16]:
from pyspark.sql.types import *

In [17]:
yt = yt.withColumn('likes',yt.likes.cast(IntegerType())).withColumn('dislikes',yt.dislikes.cast(IntegerType()))
yt.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: date (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- favor: string (nullable = false)



In [18]:
yt = yt.withColumn('favor', when(yt.likes > yt.dislikes,'good').otherwise('bad'))
yt.select('likes','dislikes','favor').show(5)

+------+--------+-----+
| likes|dislikes|favor|
+------+--------+-----+
| 57527|    2966| good|
| 97185|    6146| good|
|146033|    5339| good|
| 10172|     666| good|
|132235|    1989| good|
+------+--------+-----+
only showing top 5 rows



- `expr(SQL)` 형식으로도 가능
> - `CASE WHEN 조건식 THEN val1 ELSE val2 END`

In [19]:
yt.select('likes','dislikes',expr('CASE WHEN likes > dislikes THEN "good" ELSE "bad" END AS favor2')).show(5)

+------+--------+------+
| likes|dislikes|favor2|
+------+--------+------+
| 57527|    2966|  good|
| 97185|    6146|  good|
|146033|    5339|  good|
| 10172|     666|  good|
|132235|    1989|  good|
+------+--------+------+
only showing top 5 rows



## 3. `date/timestamp` type 다루기

- `datediff(col1,col2)`: 두 column의 시간 차이를 column으로 반환

In [20]:
yt.select('publish_time','trending_date',datediff(yt.publish_time,yt.trending_date)).show(5)

+-------------------+-------------+-------------------------------------+
|       publish_time|trending_date|datediff(publish_time, trending_date)|
+-------------------+-------------+-------------------------------------+
|2017-11-13 17:13:01|   2017-11-14|                                   -1|
|2017-11-13 07:30:00|   2017-11-14|                                   -1|
|2017-11-12 19:05:24|   2017-11-14|                                   -2|
|2017-11-13 11:00:04|   2017-11-14|                                   -1|
|2017-11-12 18:01:41|   2017-11-14|                                   -2|
+-------------------+-------------+-------------------------------------+
only showing top 5 rows



- `col.alias('str')`: column명을 str로 변경

In [21]:
yt.select('publish_time','trending_date',
          datediff(yt.publish_time,yt.trending_date).alias('diff')).show(5)

+-------------------+-------------+----+
|       publish_time|trending_date|diff|
+-------------------+-------------+----+
|2017-11-13 17:13:01|   2017-11-14|  -1|
|2017-11-13 07:30:00|   2017-11-14|  -1|
|2017-11-12 19:05:24|   2017-11-14|  -2|
|2017-11-13 11:00:04|   2017-11-14|  -1|
|2017-11-12 18:01:41|   2017-11-14|  -2|
+-------------------+-------------+----+
only showing top 5 rows



- `year(col), month(col), dayofmonth(col), hour(col), minute(col), second(col)`: timestamp 혹은 date type에서 연월일시분초를 **숫자(int)**로 반환

In [22]:
yt.select('publish_time',year(yt.publish_time).alias('year'),month(yt.publish_time).alias('month'),
         dayofmonth(yt.publish_time).alias('day'),hour(yt.publish_time).alias('hour')).show(3)

+-------------------+----+-----+---+----+
|       publish_time|year|month|day|hour|
+-------------------+----+-----+---+----+
|2017-11-13 17:13:01|2017|   11| 13|  17|
|2017-11-13 07:30:00|2017|   11| 13|   7|
|2017-11-12 19:05:24|2017|   11| 12|  19|
+-------------------+----+-----+---+----+
only showing top 3 rows



## 4. `str`을 다루는 functions

- `lstrim(col)`: str 앞 공백 제거
- `rstrim(col)`: str 뒤 공백 제거
- `trim(col)`: str 앞 뒤 공백 제거
- `upper(col)`, `lower(col)`: 대문자/소문자 변환
- `initcap(col)`: 첫 문자만 대문자, 나머지는 소문자로 변경
- `substring(col,index,length)`: col이 포함한 str의 index부터 length만큼 문자 삭제
- `len(col)`: str의 길이
- `lpad(col,length,str)`: 열의 값을 length만큼 확장하고, 좌측을 str로 채움
- `rpad(col,length,str)`: 여르이 값을 length만큼 확장하고, 우측을 str로 채움
- `concat(col1,col2)`: col1과 col2의 값을 결합
- `concat_ws(sep,col1,col2)`: col1과 col2를 sep로 구분하여 결합
- `array_contain(col,val)`: col 값이 arr-like(list, tuple 등)일 때, 해당 값들 중 val을 포함하는 경우, True
- `array_distinct(col)`: col 값이 array-like일 때, 중복된 값 제거
- `array_remove(col,val)`: col 값이 array-like일 때, val을 포함할 경우, 해당 값에서 삭제
- `split(col,str)`: col 값을 str 기준으로 분리

In [23]:
yt.select('title',lower(yt.title), initcap(yt.title)).show(3)

+--------------------+--------------------+--------------------+
|               title|        lower(title)|      initcap(title)|
+--------------------+--------------------+--------------------+
|WE WANT TO TALK A...|we want to talk a...|We Want To Talk A...|
|The Trump Preside...|the trump preside...|The Trump Preside...|
|Racist Superman |...|racist superman |...|Racist Superman |...|
+--------------------+--------------------+--------------------+
only showing top 3 rows



- `title`과 `channel_title`을 `title, [channel]: channel_title` 형태로 결합

In [31]:
yt.select('title','channel_title',
         concat_ws(', [channel]: ',yt.title,yt.channel_title).alias('title_channel')).show(3)

+--------------------+---------------+--------------------+
|               title|  channel_title|       title_channel|
+--------------------+---------------+--------------------+
|WE WANT TO TALK A...|   CaseyNeistat|WE WANT TO TALK A...|
|The Trump Preside...|LastWeekTonight|The Trump Preside...|
|Racist Superman |...|   Rudy Mancuso|Racist Superman |...|
+--------------------+---------------+--------------------+
only showing top 3 rows



- `title`을 공백 기준으로 분리
- `split`을 사용하여 문자열을 분리하면, 분리된 각 str이 list에 저장

In [33]:
yt.select('title',split(yt.title,' ').alias('sep_title')).show(3)

+--------------------+--------------------+
|               title|           sep_title|
+--------------------+--------------------+
|WE WANT TO TALK A...|[WE, WANT, TO, TA...|
|The Trump Preside...|[The, Trump, Pres...|
|Racist Superman |...|[Racist, Superman...|
+--------------------+--------------------+
only showing top 3 rows



- `title`에 `TO`가 포함되어 있는지 여부를 `True`/`False`로 나타내는 column
- `array_contains`는 col 값이 `list`일 경우 가능하므로, `split`후 실행

In [35]:
yt.select('title',array_contains(split(yt.title, ' '), 'TO').alias('contain_TO')).show(3)

+--------------------+----------+
|               title|contain_TO|
+--------------------+----------+
|WE WANT TO TALK A...|      true|
|The Trump Preside...|     false|
|Racist Superman |...|     false|
+--------------------+----------+
only showing top 3 rows

